In [ ]:
# Source: 

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# ----------------------------
# 1. Load dataset
# ----------------------------
# (Download from Kaggle: https://www.kaggle.com/mlg-ulb/creditcardfraud)
df = pd.read_csv("creditcard.csv")

print("Dataset shape:", df.shape)
print("Fraud cases:", df['Class'].sum())

# Features = transaction details, Target = fraud (1) or not (0)
X = df.drop("Class", axis=1)
y = df["Class"]

# ----------------------------
# 2. Train-test split
# ----------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.3, 
                                                    random_state=42, 
                                                    stratify=y)

# ----------------------------
# 3. Unsupervised anomaly detection (Isolation Forest)
# ----------------------------
iso = IsolationForest(contamination=0.0017, random_state=42)  # fraud ratio ~0.17%
y_pred_iso = iso.fit_predict(X_test)

# IsolationForest outputs: -1 = anomaly, 1 = normal
y_pred_iso = np.where(y_pred_iso == -1, 1, 0)

print("\nIsolation Forest Performance:")
print(confusion_matrix(y_test, y_pred_iso))
print(classification_report(y_test, y_pred_iso, digits=4))

# ----------------------------
# 4. Supervised classification (Random Forest)
# ----------------------------
rf = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("\nRandom Forest Performance:")
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf, digits=4))

# ----------------------------
# 5. Hybrid approach (Optional idea)
# ----------------------------
# You could combine both:
# - Use IsolationForest to flag suspicious transactions
# - Then pass them to RandomForest for classification


Dataset shape: (284807, 31)
Fraud cases: 492

Isolation Forest Performance:
[[85182   113]
 [  115    33]]
              precision    recall  f1-score   support

           0     0.9987    0.9987    0.9987     85295
           1     0.2260    0.2230    0.2245       148

    accuracy                         0.9973     85443
   macro avg     0.6123    0.6108    0.6116     85443
weighted avg     0.9973    0.9973    0.9973     85443


Random Forest Performance:
[[85292     3]
 [   44   104]]
              precision    recall  f1-score   support

           0     0.9995    1.0000    0.9997     85295
           1     0.9720    0.7027    0.8157       148

    accuracy                         0.9994     85443
   macro avg     0.9857    0.8513    0.9077     85443
weighted avg     0.9994    0.9994    0.9994     85443

